In [ ]:
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path
import pandas as pd

from se_analysis import project

In [ ]:
# Header labels that identify the table and define the column boundaries.
# Order matters: it is the left-to-right column order.
COLUMNS = ["Channel Number", "Type", "Description", "Data"]

# Units the LOGR reports. OCR output is snapped to this list, which repairs the
# dropped underscores. Add to it if a site uses sensors not listed here.
KNOWN_UNITS = [
    "A", "V", "W", "Wh", "%", "deg", "deg_C", "deg_F", "K",
    "W/Sqm", "Wh/Sqm", "m/s", "mm", "mm/h", "hPa", "mbar", "ohm", "Hz", "s",
]
# Collapsed form -> canonical form, so "degC"/"deg C" and a lone "C" both land
# on deg_C. Built once from KNOWN_UNITS, then extended with OCR-specific cases.
_UNIT_LOOKUP = {u.replace("_", "").replace(" ", "").lower(): u for u in KNOWN_UNITS}
_UNIT_LOOKUP.update({"c": "deg_C", "f": "deg_F", "wsqm": "W/Sqm", "whsqm": "Wh/Sqm"})

OCR_SHIM = r"""
param([Parameter(Mandatory=$true)][string]$Path, [double]$Scale = 2.0)
$ErrorActionPreference = 'Stop'
Add-Type -AssemblyName System.Runtime.WindowsRuntime
$m = ([System.WindowsRuntimeSystemExtensions].GetMethods() | Where-Object {
    $_.Name -eq 'AsTask' -and $_.GetParameters().Count -eq 1 -and
    $_.GetParameters()[0].ParameterType.Name -eq 'IAsyncOperation`1' })[0]
function Await($op, $t) { $m.MakeGenericMethod($t).Invoke($null, @($op)).GetAwaiter().GetResult() }

$null = [Windows.Media.Ocr.OcrEngine, Windows.Foundation, ContentType=WindowsRuntime]
$null = [Windows.Graphics.Imaging.BitmapDecoder, Windows.Foundation, ContentType=WindowsRuntime]
$null = [Windows.Graphics.Imaging.BitmapTransform, Windows.Foundation, ContentType=WindowsRuntime]
$null = [Windows.Storage.StorageFile, Windows.Foundation, ContentType=WindowsRuntime]
$null = [Windows.Storage.FileAccessMode, Windows.Foundation, ContentType=WindowsRuntime]

$file    = Await ([Windows.Storage.StorageFile]::GetFileFromPathAsync($Path)) ([Windows.Storage.StorageFile])
$stream  = Await ($file.OpenAsync([Windows.Storage.FileAccessMode]::Read)) ([Windows.Storage.Streams.IRandomAccessStream])
$decoder = Await ([Windows.Graphics.Imaging.BitmapDecoder]::CreateAsync($stream)) ([Windows.Graphics.Imaging.BitmapDecoder])

# Upscaling before OCR measurably improves accuracy on UI-sized text.
$tf = New-Object Windows.Graphics.Imaging.BitmapTransform
$tf.ScaledWidth  = [uint32]([math]::Round($decoder.PixelWidth  * $Scale))
$tf.ScaledHeight = [uint32]([math]::Round($decoder.PixelHeight * $Scale))
$bmp = Await ($decoder.GetSoftwareBitmapAsync(
        [Windows.Graphics.Imaging.BitmapPixelFormat]::Bgra8,
        [Windows.Graphics.Imaging.BitmapAlphaMode]::Premultiplied,
        $tf,
        [Windows.Graphics.Imaging.ExifOrientationMode]::RespectExifOrientation,
        [Windows.Graphics.Imaging.ColorManagementMode]::DoNotColorManage)
      ) ([Windows.Graphics.Imaging.SoftwareBitmap])

$eng = [Windows.Media.Ocr.OcrEngine]::TryCreateFromUserProfileLanguages()
if (-not $eng) { throw "No OCR language pack available for this user profile." }
$res = Await ($eng.RecognizeAsync($bmp)) ([Windows.Media.Ocr.OcrResult])

# Coordinates are divided back down so they refer to the original image.
$words = foreach ($line in $res.Lines) { foreach ($w in $line.Words) {
    [pscustomobject]@{ text=$w.Text
                       x=[math]::Round($w.BoundingRect.X / $Scale, 1)
                       y=[math]::Round($w.BoundingRect.Y / $Scale, 1)
                       w=[math]::Round($w.BoundingRect.Width / $Scale, 1)
                       h=[math]::Round($w.BoundingRect.Height / $Scale, 1) } } }
@{ width=$decoder.PixelWidth; height=$decoder.PixelHeight; words=@($words) } |
    ConvertTo-Json -Depth 4 -Compress
"""

In [ ]:
def ocr_words(image: Path, scale: float = 2.0) -> tuple[dict, list[dict]]:
    """Run the image through the Windows OCR engine; return (meta, words)."""
    if sys.platform != "win32":
        raise RuntimeError("This script uses the Windows OCR engine and needs Windows.")

    with tempfile.TemporaryDirectory() as tmp:
        shim = Path(tmp) / "ocr_words.ps1"
        shim.write_text(OCR_SHIM, encoding="utf-8")
        proc = subprocess.run(
            ["powershell.exe", "-NoProfile", "-ExecutionPolicy", "Bypass",
             "-File", str(shim), "-Path", str(image.resolve()), "-Scale", str(scale)],
            capture_output=True, text=True,
        )
    if proc.returncode != 0:
        raise RuntimeError(f"OCR failed:\n{proc.stderr.strip()}")

    payload = json.loads(proc.stdout)
    return {"width": payload["width"], "height": payload["height"]}, payload["words"]


def cluster_rows(words: list[dict], tol_ratio: float = 0.6) -> list[list[dict]]:
    """Group words into visual rows by y, tolerating baseline jitter.

    Tolerance scales with glyph height rather than being a fixed pixel count,
    so the same code works on screenshots taken at different resolutions or
    browser zoom levels.
    """
    if not words:
        return []
    tol = max(4.0, tol_ratio * (sum(w["h"] for w in words) / len(words)))
    rows: list[list[dict]] = []
    for word in sorted(words, key=lambda w: w["y"]):
        centre = word["y"] + word["h"] / 2
        for row in rows:
            ref = row[0]["y"] + row[0]["h"] / 2
            if abs(centre - ref) <= tol:
                row.append(word)
                break
        else:
            rows.append([word])
    for row in rows:
        row.sort(key=lambda w: w["x"])
    return rows


def find_header(rows: list[list[dict]]) -> tuple[int, dict[str, float]]:
    """Locate the header row and each column's left edge.

    Returns the row index and a {column: x} map. Multi-word headers such as
    "Channel Number" are matched on their first token.
    """
    wanted = {c: c.split()[0].lower() for c in COLUMNS}
    for idx, row in enumerate(rows):
        seen = {}
        for word in row:
            token = word["text"].strip().lower().strip(":")
            for col, first in wanted.items():
                if token == first and col not in seen:
                    seen[col] = word["x"]
        if len(seen) == len(COLUMNS):
            return idx, seen
    raise LookupError(
        "Could not find the channel-table header. Expected all of "
        f"{COLUMNS} on one line -- is this a screenshot of the Status page?"
    )


def column_bounds(header_x: dict[str, float], width: int) -> list[tuple[str, float, float]]:
    """Turn header positions into [left, right) x-ranges, split at midpoints."""
    ordered = sorted(header_x.items(), key=lambda kv: kv[1])
    bounds = []
    for i, (col, x) in enumerate(ordered):
        left = 0.0 if i == 0 else (ordered[i - 1][1] + x) / 2
        right = float(width) if i == len(ordered) - 1 else (ordered[i + 1][1] + x) / 2
        bounds.append((col, left, right))
    return bounds


def fix_ocr_word(text: str) -> str:
    """Repair digit-for-letter confusions inside otherwise alphabetic words.

    Only a digit flanked by letters on both sides is touched, so "PVT1-S0il"
    becomes "PVT1-Soil" while genuine alphanumerics like "PVT1" are left alone.
    """
    if sum(c.isalpha() for c in text) < 3:
        return text
    return re.sub(
        r"(?<=[A-Za-z])0(?=[A-Za-z])", "o",
        re.sub(r"(?<=[A-Za-z])1(?=[A-Za-z])", "l", text),
    )


NUM_RE = re.compile(r"^[-+]?(?:\d+\.?\d*|\.\d+)$")


def split_data(cell: str) -> tuple[float | None, str, str]:
    """Split a Data cell into (value, canonical units, raw units)."""
    tokens = cell.split()
    value, unit_tokens = None, []
    for token in tokens:
        cleaned = token.replace(",", "")
        if value is None and NUM_RE.match(cleaned):
            value = float(cleaned)
        else:
            unit_tokens.append(token)

    raw = " ".join(unit_tokens)
    key = raw.replace("_", "").replace(" ", "").replace("-", "").lower()
    return value, _UNIT_LOOKUP.get(key, raw), raw


def parse_screenshot(image: Path, scale: float = 2.0, fix_words: bool = True) -> pd.DataFrame:
    """Full pipeline: image -> tidy DataFrame of channels."""
    meta, words = ocr_words(image, scale)
    rows = cluster_rows(words)
    header_idx, header_x = find_header(rows)
    bounds = column_bounds(header_x, meta["width"])

    records = []
    for row in rows[header_idx + 1:]:          # everything below the header
        cells = {col: [] for col, _, _ in bounds}
        for word in row:
            centre = word["x"] + word["w"] / 2
            for col, left, right in bounds:
                if left <= centre < right:
                    cells[col].append(word["text"])
                    break

        channel = "".join(cells["Channel Number"]).strip()
        if not channel.isdigit():
            # Browser chrome, nav buttons and the page title all fail this,
            # which is exactly how they get filtered out.
            continue

        description = " ".join(cells["Description"])
        if fix_words:
            description = " ".join(fix_ocr_word(w) for w in description.split())

        value, units, units_raw = split_data(" ".join(cells["Data"]))
        records.append({
            "Channel Number": int(channel),
            "Type": " ".join(cells["Type"]).replace("PV IEC", "PV_IEC"),
            "Description": description,
            "Value": value,
            "Units": units,
            "Units (raw OCR)": units_raw,
        })

    df = pd.DataFrame(records).sort_values("Channel Number").reset_index(drop=True)
    if df.empty:
        raise LookupError("Found the header but no channel rows beneath it.")
    return df


def warn_suspect(df: pd.DataFrame) -> list[str]:
    """Flag cells a human should eyeball before trusting them."""
    notes = []
    missing = df[df["Value"].isna()]["Channel Number"].tolist()
    if missing:
        notes.append(f"no numeric value parsed for channel(s): {missing}")

    snapped = df[(df["Units"] != df["Units (raw OCR)"]) & (df["Units (raw OCR)"] != "")]
    for _, r in snapped.iterrows():
        notes.append(
            f"ch {r['Channel Number']}: units OCR'd as "
            f"{r['Units (raw OCR)']!r}, snapped to {r['Units']!r}"
        )

    unknown = df[(df["Units"] == df["Units (raw OCR)"]) & (df["Units (raw OCR)"] != "")]
    for _, r in unknown.iterrows():
        if r["Units"] not in KNOWN_UNITS:
            notes.append(
                f"ch {r['Channel Number']}: units {r['Units']!r} not in KNOWN_UNITS"
            )

    gaps = sorted(set(range(df["Channel Number"].min(), 1 + max(
        c for c in df["Channel Number"] if c < 100))) - set(df["Channel Number"]))
    if gaps:
        notes.append(f"analog channel number(s) absent from the table: {gaps}")
    return notes


In [ ]:
file_path = project("Buena Vista NM", "MET 1", "Soiling", "Screenshot 2026-08-19 132103.png")
meta, words = ocr_words(file_path, 2.0)

In [ ]:
meta

In [ ]:
words

In [ ]:
df = parse_screenshot(file_path, 2.0, fix_words=True)
df